[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


# Occupational Classification: Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models. Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.


In [2]:
# Standard library and notebook-wide utilities
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
import importlib.util
import subprocess
import sys
from urllib.request import urlopen


# Keep the previous warning policy scoped to UserWarning, not all warnings.
warnings.filterwarnings('ignore', category=UserWarning)

# Numerical arrays and tabular survey data
import numpy as np
import pandas as pd

# Notebook display configuration: improve readability without changing data.
pd.set_option('display.html.use_mathjax', False)
pd.set_option('display.max_columns', None)

def custom_style(df):
    """Format DataFrames for notebook display; source values remain unchanged."""
    return (df.style.set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#1e293b'), ('color', '#ffffff'),
            ('font-family', 'sans-serif'), ('font-size', '13px'),
            ('padding', '10px 15px'), ('text-align', 'left')
        ]},
        {'selector': 'td', 'props': [
            ('font-family', 'sans-serif'), ('font-size', '12px'),
            ('padding', '8px 15px'), ('border-bottom', '1px solid #e2e8f0')
        ]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f1f5f9')]}
    ]).set_properties(**{'background-color': '#f8fafc'}))

# Apply the HTML style to all DataFrame previews in this notebook session.
pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

# Figures and rich notebook output
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Machine learning: cross-validation and reusable pipelines
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Candidate classifiers and majority-class baseline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

# Performance measurement
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Reproducibility: downstream model evaluation uses this fixed random seed.
SEED = 42


### Load the LaBSE model

[LaBSE](https://huggingface.co/sentence-transformers/LaBSE?utm_source=chatgpt.com)

LaBSE (Language-agnostic BERT Sentence Embedding) is a multilingual model that converts text into 768-dimensional, L2-normalized vectors in a shared space. It was optimized to place translations close together, so we use it to compare workplace names written in Hebrew and English without translating them first. Because the vectors have unit length, their dot product equals cosine similarity. A high score identifies a candidate match between two workplace names. Establishing that both names refer to the same employer requires corroborating information.

The code below loads the pretrained model on the CPU. In the workplace-name analysis, we use cosine similarity to rank possible Hebrew matches for names written in English. A pair goes forward for review when its cosine similarity exceeds the adjustable threshold α. Confirming that both names refer to the same workplace requires separate evidence. 


In [31]:
# import sentence-transformers if not already installed
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2", ])
from sentence_transformers import SentenceTransformer

# Download/cache LaBSE once. No API token, SSH, or paid GPU.
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")

LaBSE loaded.


## Source data

Load the survey responses and their metadata from the two workbook sheets without altering the source values.


In [32]:
# Retrieve the original workbook, verify its contents, and load both source tables.
URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
SHA = "e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac"
raw = urllib.request.urlopen(URL, timeout=90).read()

# Load only rows and columns without missing values
data = (pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all"))

metadata = (pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any"))

# Display the complete metadata dictionary.
display(metadata)
print()

# Display the first three survey records.
display(data.head(20))
print()

,משתנה,הסבר,פורמט
0,ID,מספר רץ,INTEGER
1,YeshuvAvoda,שם יישוב עבודה,TEXT
2,ShemAvoda,שם מקום העבודה שלך,TEXT
3,SugAvoda,הפעילות העיקרית של מקום העבודה,TEXT
4,ShemMachlaka,אגף או מחלקה שהנסקר עובד,TEXT
5,SugMachlaka,הפעילות העיקרית של האגף או המחלקה שהנסקר עובד,TEXT
6,EzoAvoda,עבודה שהנסקר עושה במקום עבודה זה,TEXT
7,TeurPeula,הפעולות העיקריות שהנסקר עושה במקום עבודה,TEXT
8,TeurTafkid,מהו תואר המשרה של הנסקר במקום עבודה זה (במידה ויש),TEXT
9,MenahelEtMi,האם הנסקר מנהל או ממונה על עובדים אחרים?,INTEGER


,ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
0,1.000000,ירושלים,"למ""ס","איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף,3.000000,1.000000,1.000000,6.000000,22.000000,55.000000,2020-06-15 00:00:00,8411,2120
1,2.000000,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan,1.000000,1.000000,6.000000,22.000000,55.000000,2022-07-01 00:00:00,8610,3311
2,3.000000,באר שבע,קריית הממשלה,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,nan,nan,1.000000,1.000000,5.000000,15.000000,29.000000,2021-06-06 00:00:00,8411,3512
3,4.000000,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,nan,שומר בבית הספר,nan,nan,nan,1.000000,2.000000,3.000000,12.000000,48.000000,2024-07-01 00:00:00,8010,5414
4,5.000000,תל אביב,אמיר כהן,מטפל בקשיש במשפחה,nan,nan,מטפל בקשיש,"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ",nan,nan,1.000000,1.000000,4.000000,14.000000,42.000000,2010-01-11 00:00:00,XXXX,5322
5,6.000000,חיפה,משטרה,שוטר,nan,nan,שוטר,nan,nan,nan,1.000000,1.000000,4.000000,14.000000,35.000000,2022-05-31 00:00:00,8423,5412
6,7.000000,קרית גת,אינטל,ייצור צ'יפים למחשבים,חדר נקי,nan,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,nan,nan,1.000000,1.000000,5.000000,15.000000,65.000000,1999-07-07 00:00:00,2610,3122
7,8.000000,חיפה,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנה,nan,"מהנדס תוכנה, מנהל צוות עובדים",מפתח קוד לצ'יפים,nan,1.000000,1.000000,1.000000,6.000000,18.000000,55.000000,2000-11-10 00:00:00,7201,2512
8,9.000000,אשקלון,שופר סל,קופאית,nan,nan,קופאית,nan,nan,nan,1.000000,3.000000,2.000000,11.000000,52.000000,2022-11-11 00:00:00,4711,5230
9,10.000000,הרצליה,EMILIO,"הייטק, פיתוח תוכנה",nan,nan,מתכנת,nan,nan,nan,1.000000,1.000000,5.000000,16.000000,24.000000,2021-12-14 00:00:00,6201,2514


### Field-by-field investigation


### Analysis of ID column

The metadata defines the values within the column ID as a values of a running index.
The metadata defines column ID to be an Integer type.

What exactly does the running index identify, the respondent, the questionnaire, a submission, or a record?

The assignment mantions explicitly an framework that allaws resubmissions from the same participant within the the same servey timeline, which means the system must have a key per a single person within a single survey, and the only one can provide that in the data is an ID column.

In [33]:
# minus 1 column name record
len(data.ID) - 1

200

Conclussion: The data analysis confirms the 200 records of surveys

### Analysis of YeshuvAvoda column

Was the job location location was obtained throgh predifined categories or filled in as a free text?
The metadata states that it is stored in a text type field, but there is no mention of haw it was obtained.
From abserving the data it is clear that most likely come from dropdawn list or chousen category becouse the only iregulaarities in the column are bouth valid verssions of the same city, just one older verssion and the other is from newear national regestery verssion. 
Most likely bouth verssions still remained, so I would raise this issue with the relavent stakeholders and peers.

In [34]:
# missing values
data.YeshuvAvoda[(data.YeshuvAvoda.empty) or (data.YeshuvAvoda == None)]

Series([], Name: YeshuvAvoda, dtype: object)

Conclussion: The data analysis confirms that the fieald is a mandotory field which is also aligngs perfectly with my initial asamption.

### Analysis of ShemAvoda column

The metadata defines ShemAvoda as the name of the participant’s workplace.

How many records are empty, how many records are distinct relative to records that not empty, haw match records contain a Latin letters, and haw many records contain a mix of Hebrew and English letters in one cell?


In [35]:
shem_avoda = data.loc[data["ID"].notna(), "ShemAvoda"].astype("string").str.strip().replace("", pd.NA)

print("Empty records:", shem_avoda.isna().sum())
print("Distinct values among nonempty records:", shem_avoda.nunique(), "of", shem_avoda.count())

has_hebrew = shem_avoda.str.contains(r"[א-ת]", na=False)
has_latin = shem_avoda.str.contains(r"[A-Za-z]", na=False)
print("Records containing Hebrew letters:", has_hebrew.sum())
print("Records containing Latin letters:", has_latin.sum())
print("Records containing both Hebrew and Latin letters:", shem_avoda[has_hebrew & has_latin == True])

Empty records: 0
Distinct values among nonempty records: 82 of 200
Records containing Hebrew letters: 187
Records containing Latin letters: 14
Records containing both Hebrew and Latin letters: 139    יועצת IT עצמאית
Name: ShemAvoda, dtype: string


Conclusion: None of the 200 workplace records is empty. There are 82 distinct values among the 200 nonempty records (41%). Fourteen records contain Latin letters, and one of those also contains Hebrew letters in the same cell.

In [38]:
def transform_column_with_labse(column, alpha=0.5):
    # Prepare this column using the same text handling as the existing analysis.
    values = column.astype("string").str.strip().replace("", pd.NA)
    has_hebrew = values.str.contains(r"[א-ת]", na=False)
    has_latin = values.str.contains(r"[A-Za-z]", na=False)

    # Select full values containing Latin letters and the existing Hebrew-only values.
    latin_cells = values.loc[has_latin]
    hebrew_names = sorted(values.loc[has_hebrew & ~has_latin].unique())

    # Encode the full values with the already loaded LaBSE model.
    vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

    # Find the highest cosine similarity for each selected value.
    similarities = vectors[:len(latin_cells)] @ vectors[len(latin_cells):].T
    best_indices = similarities.argmax(axis=1)
    best_scores = similarities[np.arange(len(latin_cells)), best_indices]
    best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=latin_cells.index, dtype="string")

    # Replace the full value only when its score exceeds alpha.
    transformed_column = column.copy()
    transformed_column.loc[latin_cells.index] = best_hebrew.where(best_scores > alpha, column.loc[latin_cells.index])
    return transformed_column


# Apply the same transformation to the workplace column.
transformed_data = data.copy()
transformed_data["ShemAvoda"] = transform_column_with_labse(data["ShemAvoda"])

# Display the original and transformed workplace values.
review = data.loc[shem_avoda.loc[has_latin].index, ["ID", "ShemAvoda"]].copy()
review["Transformed"] = transformed_data.loc[review.index, "ShemAvoda"]
display(review)

,ID,ShemAvoda,Transformed
9,10.000000,EMILIO,EMILIO
14,15.000000,NVIDIA,נבידיה
37,38.000000,INTEL,אינטל
46,47.000000,NVIDIA,נבידיה
55,56.000000,COGNYTE,COGNYTE
72,73.000000,EMILIO,EMILIO
77,78.000000,NVIDIA,נבידיה
100,101.000000,INTEL,אינטל
103,104.000000,EMILIO,EMILIO
109,110.000000,NVIDIA,נבידיה


Which Hebrew workplace value does LaBSE rank first for each standalone name written in Latin letters? Adjust `alpha` below to set the minimum cosine similarity a candidate must exceed to be reviewed.


### Numerical representation of YeshuvAvoda

Complete the locality decision established above by merging `קרית גת` with `קריית גת` and applying the one-hot encoding specified in the metadata interpretation. All 200 locality values are populated, so this field requires no null replacement. Fit the category mapping on complete-label records and retain IDs solely for alignment.

In [8]:
# Select the identified survey records and the population used to fit encoders.
survey = transformed_data.loc[transformed_data["ID"].notna()].copy()
feature_fit_rows = survey["SemelMishlachSofi"].astype("string").str.replace(r"\.0$", "", regex=True).str.fullmatch(r"[0-9]{4}", na=False)

# Apply the established locality spelling correction.
locality_values = survey["YeshuvAvoda"].astype("string").str.strip().replace({"קרית גת": "קריית גת"}).to_frame("YeshuvAvoda")

# Complete the nominal locality representation.
from sklearn.preprocessing import OneHotEncoder
locality_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
locality_encoder.fit(locality_values.loc[feature_fit_rows])
locality_matrix = locality_encoder.transform(locality_values)
locality_features = pd.DataFrame.sparse.from_spmatrix(locality_matrix, index=survey.index, columns=locality_encoder.get_feature_names_out(["YeshuvAvoda"]))
print("YeshuvAvoda one-hot matrix:", locality_matrix.shape)

YeshuvAvoda one-hot matrix: (200, 20)


### TF-IDF representation of ShemAvoda

For `ShemAvoda`, retain TF-IDF without L2 normalization because workplace names can share words while differing through additional identifying terms. With ordinary term counts, an added term contributes another weighted feature without rescaling the shared words. This preserves weighted text magnitude as potential classification information. Whether that information improves prediction must be assessed against L2-normalized TF-IDF within training-fold validation.

Lowercase the text after LaBSE, retain Hebrew and Latin words and single-letter initials, and use unsmoothed IDF. The matrix below describes the current survey. Vocabulary and IDF must be fitted within each training partition when evaluating a classifier. [scikit-learn TF-IDF documentation](https://scikit-learn.org/stable/modules/feature_extraction.html#tfidf-term-weighting)


In [9]:
# Select the workplace text produced by the preceding LaBSE transformation.
shem_avoda_transformed = transformed_data.loc[transformed_data["ID"].notna(), "ShemAvoda"].astype("string").str.strip()

# Retain ordinary word counts and unsmoothed IDF without L2 normalization.
shem_avoda_vectorizer = TfidfVectorizer(lowercase=True, token_pattern=r"(?u)\b\w+\b", binary=False, sublinear_tf=False, smooth_idf=False, norm=None)
shem_avoda_vectorizer.fit(shem_avoda_transformed.loc[feature_fit_rows])
shem_avoda_tfidf = shem_avoda_vectorizer.transform(shem_avoda_transformed)

# Keep the vocabulary in the same order as the sparse matrix columns.
shem_avoda_terms = shem_avoda_vectorizer.get_feature_names_out()
print("ShemAvoda TF-IDF matrix:", shem_avoda_tfidf.shape)
print("Nonzero weights:", shem_avoda_tfidf.nnz)

ShemAvoda TF-IDF matrix: (200, 108)
Nonzero weights: 365


**Conclusion:** The workplace-name column is represented numerically by TF-IDF weights after the approved LaBSE transformation. All 200 workplace names are populated, so this field needs no null replacement. Its vocabulary and IDF are fitted on complete-label records. Classifier validation must refit these learned quantities inside each training fold.

## From workplace normalization to occupational evidence

The assignment asks for an occupation prediction from a questionnaire that can be submitted with unanswered fields. The remaining columns describe different things: the organization, the participant's own work, employment arrangements, education, age, dates, and existing classifications. Those differences determine what can be inferred from a response and what preparation it needs.

The investigation below starts from the workplace transformation above. Each field is interpreted against its metadata and the assignment, examined using the recorded answers, and then prepared according to the resulting conclusion. The original answers remain available alongside the prepared values.

In [10]:
# Continue with the survey records selected above.
preparation_audit = survey[["ID"]].copy()

# Carry forward the completed locality and workplace-name representations.
shem_avoda_features = pd.DataFrame.sparse.from_spmatrix(shem_avoda_tfidf, index=survey.index, columns="ShemAvoda__" + shem_avoda_vectorizer.get_feature_names_out())
prepared_data = survey.drop(columns=["ShemAvoda", "YeshuvAvoda"]).join(shem_avoda_features).join(locality_features)
field_encoders = {"YeshuvAvoda": locality_encoder, "ShemAvoda": shem_avoda_vectorizer}

### Analysis of SugAvoda column

The metadata defines `SugAvoda` as the workplace's main activity. This is an open description of what the organization does. Whole-answer categories would treat related phrases as unrelated labels, while TF-IDF can retain shared activity terms and the words that distinguish them.

Which recorded values describe an activity, and which provide no usable description? Inspect the answer frequencies and full values containing Latin letters before defining the treatment.

In [11]:
# Inspect the workplace activity answers after removing surrounding whitespace.
workplace_activity = survey["SugAvoda"].astype("string").str.strip().replace("", pd.NA)
display(workplace_activity.value_counts(dropna=False).rename("Records"))

# Inspect the full activity descriptions containing Latin letters.
display(workplace_activity.loc[workplace_activity.str.contains(r"[A-Za-z]", na=False)].to_frame())

SugAvoda
<NA>                 25
מסעדה                 7
בית ספר יסודי         7
חנות למכירת בגדים     7
אירוח                 6
                     ..
מכונאות               1
מלצר                  1
חבלן                  1
גננת                  1
סטטיסטיקה             1
Name: Records, Length: 79, dtype: Int64

,SugAvoda
2,מתן שירות למשתמשים באתר GOV
65,מתן שירות למשתמשים באתר GOV
139,יועצת IT


**Conclusion and preparation:** The 175 populated answers contain 78 distinct values, including workplace types, activities, and job descriptions. Twenty-five answers are blank. The assignment permits partially answered questionnaires, so these blanks provide no workplace-activity information. Filling them with a frequent activity would invent an answer, while deleting their records would discard other available information. Replace them with an empty string for TF-IDF, which contributes no terms and produces a zero text vector.

Four answers say `חסוי`. This is a withholding response rather than an activity, so it also receives an empty text input. The original answers remain in the source data. Preserve the three Latin-containing values in full and encode substantive wording with the same TF-IDF settings as `ShemAvoda`. Replace this text column in the prepared table with its term-weight columns. Fit vocabulary and IDF on complete-label records and refit them within each training fold during classifier validation. [TF-IDF documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)

In [12]:
# Replace unanswered and withheld activity descriptions with empty text.
activity_text = workplace_activity.mask(workplace_activity.eq("חסוי"), "").fillna("")

# Fit this field's vocabulary with the existing TF-IDF configuration on complete-label records.
activity_vectorizer = clone(shem_avoda_vectorizer)
activity_vectorizer.fit(activity_text.loc[feature_fit_rows])

# Transform the activity values into numerical term weights.
activity_matrix = activity_vectorizer.transform(activity_text)
activity_features = pd.DataFrame.sparse.from_spmatrix(activity_matrix, index=survey.index, columns="SugAvoda__" + activity_vectorizer.get_feature_names_out())

# Replace the original text column in the prepared table with its TF-IDF columns.
prepared_data = prepared_data.drop(columns="SugAvoda").join(activity_features)
field_encoders["SugAvoda"] = activity_vectorizer

# Report the completed transformation.
print("SugAvoda TF-IDF matrix:", activity_matrix.shape)

SugAvoda TF-IDF matrix: (200, 116)


### Analysis of ShemMachlaka column

The metadata asks for the division or department in which the participant works. Department names may combine a broad function with a specialization, so individual words can carry useful information beyond the identity of the complete answer. A blank records an unanswered question and does not establish that no department exists.

Do the observed answers form a documented category set, or do they contain varied descriptions and abbreviations? Inspect their frequencies to decide how much wording the representation must retain.

In [13]:
# Inspect department names, abbreviations, and unanswered records.
department_names = survey["ShemMachlaka"].astype("string").str.strip().replace("", pd.NA)
display(department_names.value_counts(dropna=False).rename("Records"))

ShemMachlaka
<NA>               64
שמירה               7
תמיכה בלקוחות       6
קוסמטיקה וטיפוח     6
משק                 6
                   ..
חשמלאות             1
יומנאי              1
מכונאות             1
מסעדה               1
חבלן                1
Name: Records, Length: 62, dtype: Int64

**Conclusion and preparation:** The 136 populated answers contain 61 distinct values, including department names, functions, job wording, and six Latin-containing entries. Four answers are `חסוי`, while 64 are blank. A blank department answer does not establish that no department exists, so neither a department name nor a “no department” category can be imputed from it. Use empty text to give it no term evidence. The withholding marker also supplies no department name and receives empty text. Preserve substantive names and abbreviations, and replace this field with its TF-IDF weights using the existing configuration. The source table retains every original answer.

In [14]:
# Preserve department names and abbreviations while leaving unknown names empty.
department_text = department_names.mask(department_names.eq("חסוי"), "").fillna("")

# Fit the department vocabulary using the established TF-IDF configuration.
department_vectorizer = clone(shem_avoda_vectorizer)
department_vectorizer.fit(department_text.loc[feature_fit_rows])

# Encode this field and replace its text column with numerical features.
department_matrix = department_vectorizer.transform(department_text)
department_features = pd.DataFrame.sparse.from_spmatrix(department_matrix, index=survey.index, columns="ShemMachlaka__" + department_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="ShemMachlaka").join(department_features)
field_encoders["ShemMachlaka"] = department_vectorizer

# Report the completed department-name transformation.
print("ShemMachlaka TF-IDF matrix:", department_matrix.shape)

ShemMachlaka TF-IDF matrix: (200, 77)


### Analysis of SugMachlaka column

The metadata defines `SugMachlaka` as the department’s main activity. Each row is one participant’s questionnaire. The participant may write `כנ"ל`, meaning “the same,” after the preceding answer instead of repeating its text. Here the preceding field is `ShemMachlaka`, so that row’s department name supplies the repeated answer.

Inspect the recorded activities and references before completing the text transformation.

In [15]:
# Inspect department activities and reference expressions.
department_activity = survey["SugMachlaka"].astype("string").str.strip().replace("", pd.NA)
display(department_activity.value_counts(dropna=False).rename("Records"))

SugMachlaka
<NA>                           134
כנ"ל                             8
טיפול בחולי סרטן                 4
טיפול באנשים עם בעיות            4
משחק                             4
אריזה                            3
טיפול בלקוחות עסקיים             3
תמיכה במערכות מחשוב של הבנק      3
חישובי שכר לעובדים               3
טיפול בנתוני מיקרו               2
הזמנת ציוד                       2
R&D                              2
רוקח                             2
אשראי                            2
סיעוד                            2
קבלת קהל                         2
טיפול בנתוני מאקרו               1
חשבת שכר לעובדים                 1
איסוף נתונים בשטח                1
טיפול בבקשת הלואות               1
שיווק עוגות                      1
מחקר ופיתוח                      1
מחסן                             1
משאבי אנוש                       1
IT                               1
מדע נתונים                       1
חפירות                           1
שיווק                            1
רואה חשב

**Conclusion and preparation:** The field has 134 blanks and 66 populated answers, including eight `כנ"ל` references. Copy `ShemMachlaka` from the same row into those eight references because the participant has explicitly repeated the preceding answer. Keep genuinely unanswered cells empty. Then apply the existing LaBSE method to this column with its default `alpha=0.5` before TF-IDF. The method selects Hebrew values already present in this feature.

In [16]:
# Resolve the participant's reference to the preceding answer in the same row.
department_activity_text = department_activity.copy()
department_activity_text.loc[department_activity_text.eq('כנ"ל')] = department_names
department_activity_text = department_activity_text.fillna("")

# Apply the existing LaBSE method to this column before TF-IDF.
department_activity_text = transform_column_with_labse(department_activity_text)

# Fit this field's vocabulary using the existing TF-IDF configuration.
department_activity_vectorizer = clone(shem_avoda_vectorizer)
department_activity_vectorizer.fit(department_activity_text.loc[feature_fit_rows])

# Replace department-activity text with its numerical term weights.
department_activity_matrix = department_activity_vectorizer.transform(department_activity_text)
department_activity_features = pd.DataFrame.sparse.from_spmatrix(department_activity_matrix, index=survey.index, columns="SugMachlaka__" + department_activity_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="SugMachlaka").join(department_activity_features)
field_encoders["SugMachlaka"] = department_activity_vectorizer
print("SugMachlaka TF-IDF matrix:", department_activity_matrix.shape)

SugMachlaka TF-IDF matrix: (200, 58)


### Analysis of EzoAvoda column

The metadata asks what work the participant performs. The answer can state a short occupation name or describe duties, so its words provide direct evidence for occupation classification. Missing and withheld answers contain no work description. Spelling variation requires care because changing a word changes the evidence supplied by the participant.

Inspect the recorded descriptions to distinguish usable text from non-disclosure and to identify wording that must survive preparation.

In [17]:
# Inspect work descriptions, spelling variants, and non-disclosure responses.
work_descriptions = survey["EzoAvoda"].astype("string").str.strip().replace("", pd.NA)
display(work_descriptions.value_counts(dropna=False).rename("Records"))

EzoAvoda
שומר בבית הספר                7
<NA>                          7
טבח                           7
קוסמטיקאית                    6
מנקה בית ספר אחרי הלימודים    6
                             ..
בדיקת דירוג אשראי             1
מציל בבריכה                   1
מכונאי של רכבים חשמליים       1
חבלן                          1
סימול סקרים                   1
Name: Records, Length: 85, dtype: Int64

**Conclusion and preparation:** There are 193 populated answers, seven blanks, and 84 distinct populated values. Two answers say `חסוי` and two say `לא יכול לפרט`, leaving 189 substantive descriptions. The seven blanks cannot be interpreted as unemployment because this question asks for a description of work and the assignment allows omitted answers. Use empty text for unanswered and explicitly withheld descriptions. Apply the existing LaBSE method to this column with its default `alpha=0.5`, then encode the returned descriptions with TF-IDF. No occupation is imputed from a frequent answer.

In [18]:
# Preserve work descriptions and give unanswered or withheld descriptions no text terms.
work_description_text = work_descriptions.mask(work_descriptions.isin(["חסוי", "לא יכול לפרט"]), "").fillna("")

# Apply the existing LaBSE method to this column before TF-IDF.
work_description_text = transform_column_with_labse(work_description_text)

# Fit the work-description vocabulary using the existing TF-IDF configuration.
work_description_vectorizer = clone(shem_avoda_vectorizer)
work_description_vectorizer.fit(work_description_text.loc[feature_fit_rows])

# Replace the description column with its numerical term weights.
work_description_matrix = work_description_vectorizer.transform(work_description_text)
work_description_features = pd.DataFrame.sparse.from_spmatrix(work_description_matrix, index=survey.index, columns="EzoAvoda__" + work_description_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="EzoAvoda").join(work_description_features)
field_encoders["EzoAvoda"] = work_description_vectorizer
print("EzoAvoda TF-IDF matrix:", work_description_matrix.shape)

EzoAvoda TF-IDF matrix: (200, 158)


### Analysis of TeurPeula column

The metadata asks for the main actions performed at work. Verbs and the objects of those actions distinguish duties, so retaining phrase content is more useful than assigning a separate category to every answer. Each row belongs to one participant. When the participant writes `כנ"ל`, meaning “the same,” the task answer repeats the preceding `EzoAvoda` answer in that row.

Inspect the reported tasks and reference expressions before completing the transformation.

In [19]:
# Inspect the reported tasks and the references recorded instead of task descriptions.
task_descriptions = survey["TeurPeula"].astype("string").str.strip().replace("", pd.NA)
display(task_descriptions.value_counts(dropna=False).rename("Records"))

TeurPeula
<NA>                                                 136
כנ"ל                                                  12
מפקח על תהליך ייצור צ'יפים                             4
מסבירה לאנשים לגבי מוצרים של חברת LOREAL               4
נותן תמיכה לעובדים בבנק ופוטר בעיות תקשורת             3
עוזר לחשמלאי                                           3
טכנאי מזגנים                                           3
קבלת קהל                                               3
נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ               2
מפתח קוד לצ'יפים                                       2
מאכיל ומנקה                                            2
בחינת יכולת החזר                                       2
פיתוח קוד                                              2
טיפול יום יומי בצרכים של הקשיש                         2
מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט      2
בונה מודלים של AI                                      1
עוזר לתקן צינורות                                      1
חישובי שכר לעובדים   

**Conclusion and preparation:** Of 64 populated answers, 12 are `כנ"ל` and 52 explicitly describe tasks. Another 136 answers are blank. An unanswered task field does not establish that the participant performs no tasks, so leave genuine blanks empty. For each `כנ"ל`, copy the resolved `EzoAvoda` text from the same row because the participant has stated that the answer is the same. Apply the existing LaBSE method to the completed task column with its default `alpha=0.5`, then perform TF-IDF.

In [20]:
# Copy the preceding work description where the participant wrote "the same."
task_text = task_descriptions.copy()
task_text.loc[task_text.eq('כנ"ל')] = work_description_text
task_text = task_text.fillna("")

# Apply the existing LaBSE method to this column before TF-IDF.
task_text = transform_column_with_labse(task_text)

# Fit the task vocabulary using the existing TF-IDF configuration.
task_vectorizer = clone(shem_avoda_vectorizer)
task_vectorizer.fit(task_text.loc[feature_fit_rows])

# Replace task text with numerical term weights.
task_matrix = task_vectorizer.transform(task_text)
task_features = pd.DataFrame.sparse.from_spmatrix(task_matrix, index=survey.index, columns="TeurPeula__" + task_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="TeurPeula").join(task_features)
field_encoders["TeurPeula"] = task_vectorizer
print("TeurPeula TF-IDF matrix:", task_matrix.shape)

TeurPeula TF-IDF matrix: (200, 105)


### Analysis of TeurTafkid column

The metadata asks for a job title if one exists. An unanswered optional title does not establish that the participant lacks an occupation. Titles can contain qualifiers such as deputy or manager, so collapsing a complete phrase to an arbitrary category can discard relationships between its words.

Inspect all distinct answers to distinguish titles and descriptive wording from references and placeholders.

In [21]:
# Inspect every distinct title value, including missing answers.
job_titles = survey["TeurTafkid"].astype("string").str.strip().replace("", pd.NA)
display(job_titles.value_counts(dropna=False).rename("Records"))

TeurTafkid
<NA>                                  172
כלכלן                                   6
כנ"ל                                    4
סגן מנהל אגף                            3
טכנאי מחשבים                            3
0                                       1
בונה מודלים של AI                       1
סוקר                                    1
פיתוח תכנה                              1
מנהל פיתןח של 2 מוצרים                  1
מנהלת משאבי אנוש של עובדיםן בישראל      1
יועצת IT                                1
סמנכ"ל שיווק                            1
תמיכה במחשוב                            1
מנהלת משאבי אנוש                        1
יועץ השקעות                             1
פקיד                                    1
Name: Records, dtype: Int64

**Conclusion and preparation:** There are 172 blanks and 28 populated answers. The optional title question allows a blank to reflect either no formal title or an omitted answer, so neither case justifies inventing a title. Four answers are `כנ"ל` and one is `0`. Copy the resolved `TeurPeula` answer into each reference, working from left to right within that participant’s row. Thus `טכנאית רדיותרפיה | כנ"ל | כנ"ל` becomes three occurrences of `טכנאית רדיותרפיה`. Keep genuine blanks and the zero placeholder empty, then encode the completed title text with TF-IDF.

In [22]:
# Propagate the already resolved task answer into the participant's title reference.
title_text = job_titles.copy()
title_text.loc[title_text.eq('כנ"ל')] = task_text

# Keep genuine blanks and the zero placeholder empty.
title_text = title_text.replace("0", "").fillna("")

# Fit the title vocabulary using the existing TF-IDF configuration.
title_vectorizer = clone(shem_avoda_vectorizer)
title_vectorizer.fit(title_text.loc[feature_fit_rows])

# Replace title text with numerical term weights.
title_matrix = title_vectorizer.transform(title_text)
title_features = pd.DataFrame.sparse.from_spmatrix(title_matrix, index=survey.index, columns="TeurTafkid__" + title_vectorizer.get_feature_names_out())
prepared_data = prepared_data.drop(columns="TeurTafkid").join(title_features)
field_encoders["TeurTafkid"] = title_vectorizer
print("TeurTafkid TF-IDF matrix:", title_matrix.shape)

TeurTafkid TF-IDF matrix: (200, 33)


### Analysis of MenahelEtMi column

The metadata records supervisory responsibility using three codes: 1 for employees, 2 for managers, and 3 for both. The preparation specification treats this field as a management level and assigns zero to unanswered records. Preserve that specified 0–3 representation as one numeric feature.

Count the recorded levels and missing answers to establish how many records receive the zero encoding.

In [23]:
# Count each management code and the unanswered records.
management_level = survey["MenahelEtMi"]
display(management_level.value_counts(dropna=False).sort_index().rename("Records"))

MenahelEtMi
1.0      8
2.0     11
3.0     13
NaN    168
Name: Records, dtype: int64

**Conclusion and preparation:** Codes 1, 2, and 3 occur in 8, 11, and 13 records respectively. Assign zero to the 168 unanswered records and retain the recorded levels for the other 32. Zero denotes the specified encoding of an unanswered field. Keep the original responses in the source table.

In [24]:
# Apply the specified management-level encoding.
prepared_data["MenahelEtMi"] = management_level.fillna(0).astype("int64")

### Analysis of MaamadAvoda column

The metadata defines seven employment arrangements: employee, self-employed with one or two employees, self-employed with at least three employees, self-employed without employees, cooperative member, kibbutz member, and unpaid family worker. The code numbers identify arrangements rather than measuring a quantity, so this field requires one-hot encoding.

Inspect the observed categories and unanswered records before preparing the categorical input.

In [25]:
# Count the employment arrangements represented in the sample.
employment_status = survey["MaamadAvoda"]
display(employment_status.value_counts(dropna=False).sort_index().rename("Records"))

MaamadAvoda
1.0    177
2.0      1
4.0     10
6.0     10
NaN      2
Name: Records, dtype: int64

**Conclusion and preparation:** The sample contains 177 employees, one self-employed participant with one or two employees, ten self-employed participants without employees, and ten kibbutz members. Two answers are blank. A blank establishes no employment arrangement, so assigning the majority category would incorrectly label the participant as an employee. Preserve missing as an unanswered category and one-hot encode the employment codes. Fit the mapping on complete-label records and use the same fitted mapping when transforming other records.

In [26]:
# Keep unanswered employment status distinct from every recorded arrangement.
employment_categories = employment_status.astype("Int64").astype("string").fillna("Missing")

# Fit the specified nominal encoding on complete-label records.
employment_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
employment_encoder.fit(employment_categories.loc[feature_fit_rows].to_frame("MaamadAvoda"))

# Replace employment codes with their one-hot representation.
employment_matrix = employment_encoder.transform(employment_categories.to_frame("MaamadAvoda"))
employment_features = pd.DataFrame.sparse.from_spmatrix(employment_matrix, index=survey.index, columns=employment_encoder.get_feature_names_out(["MaamadAvoda"]))
prepared_data = prepared_data.drop(columns="MaamadAvoda").join(employment_features)
field_encoders["MaamadAvoda"] = employment_encoder
print("MaamadAvoda one-hot matrix:", employment_matrix.shape)

MaamadAvoda one-hot matrix: (200, 4)


### Analysis of MakorSachar column

The metadata defines four salary sources and restricts this question to employees and cooperative members, corresponding to `MaamadAvoda` codes 1 and 5. Salary source is nominal and requires one-hot encoding. A blank can mean that the question does not apply or that an applicable answer was omitted. Its treatment therefore depends on the questionnaire's explicit routing condition.

Count salary-source responses and apply that condition to distinguish unanswered, non-applicable, and contradictory records.

In [27]:
# Count the salary sources recorded in this field.
salary_source = survey["MakorSachar"]
display(salary_source.value_counts(dropna=False).sort_index().rename("Records"))

# Apply the metadata's eligibility rule to interpret the unanswered salary-source cells.
salary_status_known = survey["MaamadAvoda"].notna()
salary_eligible = survey["MaamadAvoda"].isin([1, 5])
salary_ineligible = salary_status_known & ~salary_eligible
display(pd.Series({"Eligible with an answer": int((salary_eligible & salary_source.notna()).sum()), "Ineligible without an answer": int((salary_ineligible & salary_source.isna()).sum()), "Ineligible with an answer": int((salary_ineligible & salary_source.notna()).sum()), "Unknown eligibility without an answer": int((~salary_status_known & salary_source.isna()).sum())}))

# Retain the recorded combinations that contradict the routing rule.
salary_eligibility_issues = survey.loc[salary_ineligible & salary_source.notna(), ["ID", "MaamadAvoda", "MakorSachar"]]
display(salary_eligibility_issues)

MakorSachar
1.0    144
2.0     26
3.0     11
4.0      5
NaN     14
Name: Records, dtype: int64

Eligible with an answer                  177
Ineligible without an answer              12
Ineligible with an answer                  9
Unknown eligibility without an answer      2
dtype: int64

,ID,MaamadAvoda,MakorSachar
33,34.000000,4.000000,4.000000
48,49.000000,4.000000,1.000000
111,112.000000,4.000000,1.000000
139,140.000000,4.000000,1.000000
165,166.000000,4.000000,1.000000
175,176.000000,4.000000,1.000000
186,187.000000,4.000000,1.000000
189,190.000000,2.000000,1.000000
197,198.000000,4.000000,1.000000


**Conclusion and preparation:** All 177 eligible records contain an answer. Twelve ineligible records leave the field blank, nine ineligible records nevertheless contain a source code, and two records have both employment status and salary source unanswered. Apply `Not applicable` to the 21 ineligible records in accordance with the routing rule and retain their original answers for review. Keep the two unanswered records with unknown eligibility as `Missing`. Neither state identifies a payer, so neither should be replaced with the most frequent salary source. One-hot encode the resulting categories within this section.

In [28]:
# Preserve recorded salary sources and distinguish unanswered from non-applicable questions.
salary_categories = salary_source.astype("Int64").astype("string").fillna("Missing")
salary_categories = salary_categories.mask(salary_ineligible, "Not applicable")

# Fit the nominal salary-source encoder on complete-label records.
salary_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
salary_encoder.fit(salary_categories.loc[feature_fit_rows].to_frame("MakorSachar"))

# Replace salary-source text categories with their one-hot features.
salary_matrix = salary_encoder.transform(salary_categories.to_frame("MakorSachar"))
salary_features = pd.DataFrame.sparse.from_spmatrix(salary_matrix, index=survey.index, columns=salary_encoder.get_feature_names_out(["MakorSachar"]))
prepared_data = prepared_data.drop(columns="MakorSachar").join(salary_features)
field_encoders["MakorSachar"] = salary_encoder
print("MakorSachar one-hot matrix:", salary_matrix.shape)

MakorSachar one-hot matrix: (200, 5)


### Analysis of TeudaGvoha column

The metadata records the highest certificate or degree. Codes 1–7 cover school qualifications through a doctorate, while 8 means another certificate and 9 means no certificate. The entire code list therefore cannot be treated as one increasing numeric measure. Use one-hot encoding to preserve the qualification categories.

Inspect their frequencies and the number of unanswered qualifications before constructing the categorical input.

In [29]:
# Count the recorded qualification categories.
qualifications = survey["TeudaGvoha"]
display(qualifications.value_counts(dropna=False).sort_index().rename("Records"))

TeudaGvoha
1.0     3
2.0    12
3.0    57
4.0    58
5.0    25
6.0    39
7.0     4
NaN     2
Name: Records, dtype: int64

**Conclusion and preparation:** The 198 populated answers use qualification codes 1–7, and two answers are blank. Code 9 explicitly means no certificate, so replacing a blank with that code would claim something the participant did not report. Retain an unanswered category and one-hot encode the qualification codes. This preserves the distinction between a reported qualification and unavailable information without imposing a numeric distance between certificates.

In [30]:
# Keep an unanswered qualification separate from the explicit no-certificate code.
qualification_categories = qualifications.astype("Int64").astype("string").fillna("Missing")

# Fit the qualification categories on complete-label records.
qualification_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
qualification_encoder.fit(qualification_categories.loc[feature_fit_rows].to_frame("TeudaGvoha"))

# Replace qualification codes with one-hot features.
qualification_matrix = qualification_encoder.transform(qualification_categories.to_frame("TeudaGvoha"))
qualification_features = pd.DataFrame.sparse.from_spmatrix(qualification_matrix, index=survey.index, columns=qualification_encoder.get_feature_names_out(["TeudaGvoha"]))
prepared_data = prepared_data.drop(columns="TeudaGvoha").join(qualification_features)
field_encoders["TeudaGvoha"] = qualification_encoder
print("TeudaGvoha one-hot matrix:", qualification_matrix.shape)

TeudaGvoha one-hot matrix: (200, 7)


### Analysis of shnotlimud column

The metadata defines years of schooling as an integer measurement. The preparation specification sets an admissible range of 8–80 years and converts values outside that range to missing. Apply this explicit rule to the recorded measurement, then learn any replacement value from training records.

Inspect the observed distribution to identify the entries affected by the specified range.

In [31]:
# Inspect recorded schooling durations and missing measurements.
school_years = survey["shnotlimud"]
display(school_years.value_counts(dropna=False).sort_index().rename("Records"))

shnotlimud
7.0      2
8.0      1
10.0     6
11.0     5
12.0    53
13.0    25
14.0    24
15.0    30
16.0    11
17.0    10
18.0    16
20.0     7
22.0     8
NaN      2
Name: Records, dtype: int64

**Conclusion and preparation:** There are 198 recorded values between 7 and 22 years and two blank answers. The two seven-year values fall below the specified 8–80 preparation range and become missing, giving four missing measurements. Zero would assert no schooling and would violate that range. Use the median of admissible complete-label training records as a central replacement without using the occupation label to estimate schooling. Fit the replacement within each training fold during validation. Preserve the reported values in the source data. [Median imputation](https://scikit-learn.org/stable/modules/impute.html#univariate-feature-imputation)

In [32]:
# Apply the specified schooling range.
schooling_values = school_years.where(school_years.between(8, 80)).to_frame("shnotlimud")

# Learn a median replacement from admissible complete-label training records.
from sklearn.impute import SimpleImputer
schooling_imputer = SimpleImputer(strategy="median")
schooling_imputer.fit(schooling_values.loc[feature_fit_rows])

# Complete this numeric column without adding indicator features.
prepared_data["shnotlimud"] = schooling_imputer.transform(schooling_values).ravel()
field_encoders["shnotlimud"] = schooling_imputer
print("Schooling values replaced:", int(schooling_values["shnotlimud"].isna().sum()))
print("Training median:", schooling_imputer.statistics_[0])

Schooling values replaced: 4
Training median: 14.0


### Analysis of Gil column

The metadata defines age as an integer measurement. The preparation specification admits ages from 18 to 120 and converts values outside that range to missing. Age remains numeric because differences between recorded ages have a quantitative meaning.

Inspect its distribution to determine whether the rule removes any observed measurements and how many replacements will be needed.

In [33]:
# Inspect the observed age distribution and unanswered records.
age = survey["Gil"]
display(age.value_counts(dropna=False).sort_index().rename("Records"))

Gil
22.0     7
23.0     4
24.0     5
25.0    16
27.0     4
28.0     6
29.0     3
30.0     2
31.0    15
32.0    15
33.0     4
35.0    14
37.0     6
38.0     5
39.0     1
40.0     8
42.0    10
43.0     1
44.0    10
45.0     6
46.0     8
48.0     8
52.0     8
55.0    15
58.0     5
59.0     1
60.0     2
65.0     4
78.0     5
NaN      2
Name: Records, dtype: int64

**Conclusion and preparation:** All 198 recorded ages, ranging from 22 to 78, satisfy the specified 18–120 range. Two answers are blank. Zero is not a plausible age under this specification, and dropping those records would discard their remaining information. Replace the two blanks with the median age fitted on complete-label training records. This supplies a central numeric input without claiming to recover the participant's actual age. Refit the replacement within each training fold during validation.

In [34]:
# Retain ages within the specified range.
age_values = age.where(age.between(18, 120)).to_frame("Gil")

# Learn the median replacement using complete-label training records.
age_imputer = SimpleImputer(strategy="median")
age_imputer.fit(age_values.loc[feature_fit_rows])

# Complete this numeric column without adding indicator features.
prepared_data["Gil"] = age_imputer.transform(age_values).ravel()
field_encoders["Gil"] = age_imputer
print("Ages replaced:", int(age_values["Gil"].isna().sum()))
print("Training median:", age_imputer.statistics_[0])

Ages replaced: 2
Training median: 37.0


### Preparation of TarichHathala column

The metadata identifies the date the participant started the current role. The useful measurement is years on the job. Under the assignment's immediate-processing interpretation, tenure runs from that starting date to the processing date. Because no survey or last-update date is supplied, use the current execution date as that reference.

Record the reference date so the calculation is reproducible and state this substitution in the presentation. The model receives elapsed years rather than the calendar date.

In [35]:
# Parse the recorded role-start dates.
start_dates = pd.to_datetime(survey["TarichHathala"])

# Use the current processing date because the survey date is unavailable.
reference_date = pd.Timestamp.now(tz="UTC").normalize().tz_localize(None)
years_on_job = (reference_date - start_dates).dt.days / 365.25

# Report the reference date and the resulting tenure measurements.
print("Tenure reference date:", reference_date.date())
display(years_on_job.agg(["count", "min", "max", "median"]).rename("Years on the job"))

Tenure reference date: 2026-11-06


count     200.000000
min         0.418891
max        27.419576
median      4.599589
Name: Years on the job, dtype: float64

**Conclusion and preparation:** All 200 records yield a tenure value. Convert elapsed days to years using 365.25 days per year and retain the resulting continuous measurement. In the prepared table, `TarichHathala` contains years on the job, while the original table retains the starting date. The execution date substitutes for the unavailable survey date and must accompany the reported results.

In [36]:
# Replace the date with elapsed years in the prepared table.
prepared_data["TarichHathala"] = years_on_job

# Preserve the calculation reference for the saved results and presentation.
tenure_reference_date = reference_date.date().isoformat()

### Analysis of SemelAnafSofi column

The metadata defines a four-character industry classification. Its digits identify categories in a hierarchy rather than a numeric magnitude, and `X` denotes incomplete coding. Leading zeros therefore belong to the code representation.

Inspect the short stored values against Excel's number format before restoring a zero. Preserve partial codes. The preparation specification permits this field as a predictor only when the code is available at prediction time, which the supplied assignment does not establish.

In [37]:
# Read the stored industry codes without treating them as numeric quantities.
industry_codes = survey["SemelAnafSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)

# Inspect Excel formatting for the short numeric codes.
from openpyxl import load_workbook
source_workbook = load_workbook(io.BytesIO(raw), read_only=True, data_only=True)
source_sheet = source_workbook["Survey Dataset"]
industry_position = list(next(source_sheet.values)).index("SemelAnafSofi") + 1
short_industry_codes = industry_codes.str.fullmatch(r"[0-9]{1,3}", na=False)
industry_storage = survey.loc[short_industry_codes, ["ID", "SemelAnafSofi"]].copy()
industry_storage["Excel format"] = [source_sheet.cell(int(index) + 2, industry_position).number_format for index in industry_storage.index]
display(industry_storage)
source_workbook.close()

# Inspect the distinct partially coded values.
display(industry_codes.loc[industry_codes.str.contains("X", na=False)].value_counts().rename("Records"))

,ID,SemelAnafSofi,Excel format
29,30.000000,123,0000
61,62.000000,123,0000
92,93.000000,123,0000
124,125.000000,123,0000
150,151.000000,124,0000


SemelAnafSofi
XXXX    6
851X    1
852X    1
3XXX    1
Name: Records, dtype: Int64

**Conclusion and preparation:** Five numeric values use Excel's `0000` format and require a leading zero to reproduce their four-position codes. Nine records contain `X`: six `XXXX` and one each of `851X`, `852X`, and `3XXX`. These are partial classifications, not empty cells, so preserve every known and unresolved position. Restore leading zeros and one-hot encode the resulting categorical values. Keep the industry matrix separate from the primary predictors because its availability before occupation prediction is not established.

In [38]:
# Restore numeric code width while preserving incomplete codes.
industry_values = industry_codes.where(~industry_codes.str.fullmatch(r"[0-9]{1,4}", na=False), industry_codes.str.zfill(4)).to_frame("SemelAnafSofi")

# Fit a categorical representation of industry codes on complete-label records.
industry_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
industry_encoder.fit(industry_values.loc[feature_fit_rows])
industry_matrix = industry_encoder.transform(industry_values)
industry_features = pd.DataFrame.sparse.from_spmatrix(industry_matrix, index=survey.index, columns=industry_encoder.get_feature_names_out(["SemelAnafSofi"]))

# Keep the numerical industry representation separate until its prediction-time availability is established.
prepared_data = prepared_data.drop(columns="SemelAnafSofi")
field_encoders["SemelAnafSofi"] = industry_encoder
print("SemelAnafSofi one-hot matrix:", industry_matrix.shape)

SemelAnafSofi one-hot matrix: (200, 42)


### Analysis of SemelMishlachSofi column

The assignment identifies this four-character occupation code as the target. Codes containing `X` were not fully classified by the professional coders. They cannot supply complete labels for supervised training or full-code accuracy, but their known digits must be retained alongside later predictions.

Count complete labels, partial patterns, and records per complete class to establish the training population and its class support.

In [39]:
# Preserve the occupation codes as strings, including unresolved digits.
occupation_codes = survey["SemelMishlachSofi"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
occupation_codes = occupation_codes.where(~occupation_codes.str.fullmatch(r"[0-9]{1,4}", na=False), occupation_codes.str.zfill(4))

# Separate complete targets and inspect support for each class.
complete_target = occupation_codes.str.fullmatch(r"[0-9]{4}", na=False)
class_counts = occupation_codes.loc[complete_target].value_counts()
display(class_counts.rename("Records"))
display(occupation_codes.loc[~complete_target].value_counts().rename("Incomplete records"))
display(pd.Series({"Complete labels": int(complete_target.sum()), "Incomplete labels": int((~complete_target).sum()), "Complete classes": class_counts.size, "Singleton classes": int(class_counts.eq(1).sum())}))

SemelMishlachSofi
5223    14
9112    12
2512    10
5120     7
2511     7
5414     7
5412     7
5230     6
5322     6
2631     6
5142     6
9211     5
8183     5
2342     5
5131     5
2635     5
3512     4
3122     4
2514     4
7414     4
4313     4
2655     4
2120     3
3211     3
7127     3
2131     3
8212     2
7412     2
3323     2
3513     2
9412     2
2142     2
9329     2
2262     2
4211     2
3311     1
2113     1
4227     1
3354     1
5153     1
3332     1
2431     1
7126     1
1212     1
1330     1
1345     1
1221     1
2411     1
2423     1
7522     1
8342     1
2330     1
5222     1
2412     1
3334     1
7413     1
3312     1
5419     1
4413     1
Name: Records, dtype: Int64

SemelMishlachSofi
1XXX    4
XXXX    4
335X    1
Name: Incomplete records, dtype: Int64

Complete labels      191
Incomplete labels      9
Complete classes      59
Singleton classes     24
dtype: int64

**Conclusion and preparation:** The 191 complete labels cover 59 classes, including 24 represented by one record. Nine records have partial labels: four `1XXX`, four `XXXX`, and one `335X`. Use complete codes as categorical targets and reserve the nine partial records for prediction. Preserve known prefixes for comparison with the eventual predicted codes. Singleton classes constrain validation coverage and must remain visible in evaluation reporting.

In [40]:
# Retain the target separately from every predictor.
prepared_data["SemelMishlachSofi"] = occupation_codes
training_ids = survey.loc[complete_target, "ID"].astype("int64")
prediction_ids = survey.loc[~complete_target, "ID"].astype("int64")
y_train = occupation_codes.loc[complete_target].copy()
partial_targets = occupation_codes.loc[~complete_target].copy()

# Preserve the known part of each incomplete target for the prediction output.
known_prefixes = partial_targets.str.split("X").str[0]
display(pd.DataFrame({"ID": prediction_ids, "Recorded target": partial_targets, "Known prefix": known_prefixes}))

,ID,Recorded target,Known prefix
17,18,1XXX,1
44,45,XXXX,
49,50,1XXX,1
80,81,1XXX,1
107,108,XXXX,
112,113,1XXX,1
141,142,XXXX,
164,165,335X,335
192,193,XXXX,


## Prepared inputs

Each predictor has now been transformed within its own section. The primary input contains TF-IDF term weights, nominal one-hot encodings, the specified management level, completed schooling and age measurements, and years on the job. ID remains a record key and the occupation code remains the target. Industry is encoded separately because its availability before occupation prediction is not established. No missingness indicator features are added.

The following matrices use complete-label records to fit numerical encoders. Later classifier validation must start from source records and refit every learned transformation within the training fold, including workplace matching candidates, vocabulary, IDF, categorical mappings, and numerical replacements.

In [41]:
# Separate the completed numeric predictors from the identifier and target.
from scipy import sparse
predictor_fields = prepared_data.columns.drop(["ID", "SemelMishlachSofi"]).tolist()
X_train = prepared_data.loc[complete_target, predictor_fields].copy()
X_prediction = prepared_data.loc[~complete_target, predictor_fields].copy()

# Store the primary numerical representation as sparse matrices.
training_matrix = sparse.csr_matrix(X_train.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
prediction_matrix = sparse.csr_matrix(X_prediction.astype(pd.SparseDtype("float64", 0)).sparse.to_coo())
feature_names = np.asarray(predictor_fields)
industry_training_matrix = industry_matrix[complete_target.to_numpy()]
industry_prediction_matrix = industry_matrix[(~complete_target).to_numpy()]

# Report the prepared populations and feature dimensions.
display(pd.DataFrame({"Records": [len(X_train), len(X_prediction)], "Numerical features": [training_matrix.shape[1], prediction_matrix.shape[1]]}, index=["Complete target", "Incomplete target"]))

,Records,Numerical features
Complete target,191,695
Incomplete target,9,695


### Save the completed preparation

Save the numerical inputs, fitted field encoders, original workplace transformation output, and salary-routing discrepancies. The reference date accompanies the tenure calculation. These artifacts preserve completed preparation while classifier and representation selection remain a separate evaluation step.

In [42]:
# Save the prepared values and numerical matrices.
import joblib
import sklearn
output = Path("data_preparation_results")
output.mkdir(exist_ok=True)
prepared_data.to_csv(output / "prepared_survey.csv", index=False, encoding="utf-8-sig")
pd.concat([training_ids.rename("ID"), X_train, y_train.rename("SemelMishlachSofi")], axis=1).to_csv(output / "training_inputs.csv", index=False, encoding="utf-8-sig")
pd.concat([prediction_ids.rename("ID"), X_prediction, partial_targets.rename("Recorded_target"), known_prefixes.rename("Known_prefix")], axis=1).to_csv(output / "prediction_inputs.csv", index=False, encoding="utf-8-sig")
sparse.save_npz(output / "training_features.npz", training_matrix)
sparse.save_npz(output / "prediction_features.npz", prediction_matrix)
sparse.save_npz(output / "industry_training_features.npz", industry_training_matrix)
sparse.save_npz(output / "industry_prediction_features.npz", industry_prediction_matrix)

# Save feature mappings, source discrepancies, and field encoders.
pd.Series(feature_names, name="Feature").to_csv(output / "feature_names.csv", index=False, encoding="utf-8-sig")
pd.Series(industry_encoder.get_feature_names_out(["SemelAnafSofi"]), name="Feature").to_csv(output / "industry_feature_names.csv", index=False, encoding="utf-8-sig")
salary_eligibility_issues.to_csv(output / "preparation_audit.csv", index=False, encoding="utf-8-sig")
review.to_csv(output / "workplace_transformations.csv", index=False, encoding="utf-8-sig")
joblib.dump(field_encoders, output / "feature_encoders.joblib")

# Record the preparation choices and tenure reference used in this execution.
manifest = {"records": len(survey), "training_records": len(X_train), "prediction_records": len(X_prediction), "features": len(feature_names), "training_ids": training_ids.tolist(), "prediction_ids": prediction_ids.tolist(), "tfidf_normalization": shem_avoda_vectorizer.norm, "tfidf_smooth_idf": False, "tenure_reference_date": tenure_reference_date, "tenure_definition": "Elapsed days from role start to the current processing date divided by 365.25.", "tenure_assumption": "Immediate processing. The current date substitutes for the unavailable survey or last-update date.", "management_missing_value": 0, "schooling_range": [8, 80], "age_range": [18, 120], "added_missingness_indicators": False, "industry_in_primary_model": False, "classifier_trained": False, "sklearn_version": sklearn.__version__, "pandas_version": pd.__version__}
(output / "preparation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Prepared numerical inputs and field encoders saved.")

Prepared numerical inputs and field encoders saved.


### Classifier evaluation

Use the classifier comparison framework from the [partial solution](https://github.com/shulruferor-sudo/Advance-Data-Science/tree/main/project_notebooks/survey) with this preparation. Compare TF-IDF with and without L2 normalization on identical training folds. Apply the selected configuration consistently to the text fields. Test adding one-hot K-means membership using L2-normalized text vectors and retain the addition only if it improves validation. Keep only the winning representation in the final fitted code, feature artifacts, and presentation. No normalization or clustering winner has been measured in this preparation stage.